# Argon A-to-Z

This tutorial aims to take the user from no familiarity with MDMC to enough competence that they can create a simple simulation and run a refinement, by walking through a simulation and refinement for the liquid argon data from [van Well et al. (1985)](https://doi.org/10.1103/PhysRevA.31.3391).  For more details on specific parts of this, please see the how-to guides!

If you'd like to learn more about a specific object, use the Python `help()` command - for example `help(Atom)` provides information on the MDMC `Atom`.

We first import all of the objects we require, and set some environmental variables.

In [ ]:
import os
import copy

import numpy as np

from MDMC.control import Control
from MDMC.MD import Atom, NonBonded, Simulation, Universe
from MDMC.MD.interactions import NonBondedForce
from MDMC.readers.observables.xml_SQw import XML_SQw
from MDMC.refinement.FoM.FoM_abs import ObservablePair
from MDMC.trajectory_analysis.observables.mdanse_observable import MDANSEObservable, get_default_mdanse_settings


### Setting up a configuration and simulation

We now build our `Universe`. For this tutorial we're creating a cube of argon-36 atoms; our side length is 23.0668Å (angstrom), and the atoms fill it with a density of 0.0176 atoms per cubic angstrom. This simulation matches the experimental data that we'd like to refine against - you will see more about this below.

Atoms are created using MDMC `Atom` objects, and then the universe is filled with them via the `universe.fill` method.

In [ ]:
universe = Universe(dimensions=23.0668)

Ar = Atom('Ar[36]', charge=0.)
universe.fill(Ar, num_density=0.0176)

Note that at this point, there are no interaction forces between the argon atoms! The simulation doesn't know how these atoms should interact with each other. In the cell below an appropriate (for argon) force-field interaction potential is defined; we use a dispersive interaction with potential energy calculated by the [Lennard-Jones potential](https://en.wikipedia.org/wiki/Lennard-Jones_potential). This interaction has two parameters, epsilon and sigma, which determine the strength of the potential energy between atoms.

In [ ]:
NonBondedForce(
        universe,
        Ar.atom_type,
        cutoff=10.0,
        ewald=1e-6,
        function=NonBonded(charge=0.0, epsilon=1.0, sigma=3.0)
    )

A `cutoff` distance, past which atoms do not interact, is chosen arbitrarily (see `help(Dispersion)` for more info). A [rule of thumb for Lennard-Jones](https://en.wikipedia.org/wiki/Lennard-Jones_potential#Lennard-Jones_truncated_&_shifted_(LJTS)_potential) is to pick `cutoff=2.5*sigma`. The value for argon is recommended to be between 8 and 12 ang. Ideally, and for any system you want to pick at value of the `cutoff` which is small while not compromising accuracy. For this system, picking a value between 8 and 12 ang is found to give near identical results to the experimental data.


### Refining our data

Our simulation is now fully set up. Now we need some data to which we fit our simulation; here we use an experimental [dynamic structure factor](https://en.wikipedia.org/wiki/Dynamic_structure_factor) $S(Q, \omega)$ for liquid argon. We call these dynamical properties an 'observable'.

In [ ]:
# exp_datasets is a list of dictionaries with one dictionary per experimental
# dataset
# Dataset from: van Well et al. (1985). Physical Review A, 31(5), 3391-3414
# resolution is None as the original author already accounted for instrument resolution
exp_datasets = [{'file_name':'data/Well_s_q_omega_Ar_data.xml',
                 'type':'SQw',
                 'reader':'xml_SQw',
                 'weight':1.,
                 'auto_scale':True,
                 'resolution':800}]

start_params = get_default_mdanse_settings("SQw")

data_parser = XML_SQw('data/Well_s_q_omega_Ar_data.xml')

exp_observable = MDANSEObservable(mdanse_job_type="SQw")
exp_observable.read_from_file(data_parser)
md_observable = MDANSEObservable(mdanse_job_type="SQw")
md_observable.origin = 'MD'
md_observable.independent_variables = copy.deepcopy(
    exp_observable.independent_variables)

observable_pair = ObservablePair(exp_obs=exp_observable,
                                 MD_obs=md_observable,
                                 weight=1.0,
                                 rescale_factor=1.0,
                                 auto_scale=True)

At this stage we check and adjust the analysis parameters.

In [ ]:
job_settings = md_observable.initial_parameters()
print("Original parameters:")
for name, value in job_settings.items():
    print(f"{name}: {value}")

new_settings = {
    "running_mode": ("multicore", -8)
}
md_observable.set_parameters(new_settings)

We can use the observables to check if the current parameters of the simulation and the observable calculation produce a result that covers the range of the experimental data.

In [ ]:
for name, axis in exp_observable.independent_variables.items():
    print(f"{name}: {axis}")

TIME_STEP_FS = 5.0
FRAME_STEP = 15
CORR_FRAMES = 200
TOTAL_FRAMES = 10000

for output_axis in md_observable.predict_output(time_step = TIME_STEP_FS,
                                                frame_step = FRAME_STEP,
                                                total_frames= TOTAL_FRAMES,
                                                correlation_frames = CORR_FRAMES,
                                                universe=universe):
    print(f"{output_axis[0]}: {output_axis[1]} {output_axis[2]}")

Next (and before starting the refinement), we set up the Simulation, which contains:
- the `Universe` that the simulation is based on;
- the MD engine used to run the simulations;
- the time-step and trajectory step which we just determined; 
- the temperature of the Universe (used to calculate atom velocity);


In [ ]:
# MD Engine setup
simulation = Simulation(universe,
                        engine="openmm",
                        time_step=TIME_STEP_FS,
                        temperature=120.,
                        traj_step=FRAME_STEP,
                        )

md_observable.set_parameters({"correlation_frames": CORR_FRAMES})

We then minimize and equilibrate the simulation; minimising the simulation avoids it getting 'stuck' in local minima, and equilibration runs the simulation until it reaches a state with a physically feasible temperature/energy distribution. This ensures our simulation isn't affected by the initial arrangement of the atoms.

In [ ]:
# Energy Minimization and equilibration
simulation.run(n_steps=10000, equilibration=True)

We then need to create our parameters to fit against. In this case, we take all the universe parameters (which here are just the sigma and epsilon values that the Lennard-Jones potential depends on). Note that above when we set our initial `LennardJones` function in the `Dispersion` object, the epsilon and sigma were our "initial guesses" that the refinement will start from.

We also set constraints for our fitting parameters; we bound our sigma values to be between 2.8 and 3.8, and our epsilon to be between 0.6 and 1.4.

In [ ]:
fit_parameters = universe.parameters
fit_parameters['sigma'].constraints = [2.7,3.8]
fit_parameters['epsilon'].constraints = [0.5, 1.5]

Now we create our `Control` object. This object oversees the refinement; it brings the simulation, dataset, and the fitting parameters together, and then does the following:
1. Run the simulation with the current parameters.
2. Calculate the simulated observable from the simulation trajectory.
3. Compare it to the experimental observable.
4. Use a minimizer (optimisation process) to refine the parameters, bringing them closer to the experimental observable.
5. Repeat with new parameters.

The minimizer we're using in this tutorial is CMA-ES (Covariance Matrix Adaptation Evolution Strategy). During the optimisation procedure, both the MD trajectory and the simulated observable can be dumped to a file for further examination by loading it into an external package like MDANSE. File dumping can be set with the `file_dump` parameters seen in the code below. Each parameter controls how often the files are dumped, what their file names or locations will be:
- `file_dump_frequency`: Defines how often the trajectory should be dumped to a H5MD file. Options available are "best" (only save the trajectory with the lowest FoM), "none" (don't save any trajectories), and "every" (save every trajectory).
- `file_dump_extent`: Which files should be written out in the dump. Options available are "traj" (H5MD trajectory file), "obs" (MDA MDANSE observable file), and "both" (both the H5MD trajectory and MDA observable files).
- `file_dump_loc`: Location that the H5MD and/or MDA files should be stored at.
- `file_dump_timestamped`: Whether a time stamp should be added to the output file names.
- `file_dump_prefix`: The name the dumped H5MD and/or MDA file should be.

In [ ]:
control = Control(simulation=simulation,
                  exp_datasets=exp_datasets,
                  fit_parameters=fit_parameters,
                  observable_pairs= [observable_pair],
                  equilibration_steps=9000,
                  MD_steps=TOTAL_FRAMES,
                  cont_slicing=True,
                  data_printer='ipython',
                  file_dump_extent="all",
                  file_dump_frequency="every",
                  FoM_options={'error': 'none'},
                  file_dump_prefix="mdmc_argon_mdanse",
                  conv_tol = 1e-6,)


Now that the dataset has been specified, and used to configure various processes and parameters, the system can be equilibrated.

In [ ]:
# Energy Minimization and equilibration
control.minimize(n_steps=5000)
control.equilibrate(n_steps=15000)

The number of `MD_steps` specified must be large enough to allow for statistically reasonable calculation of all observables. This depends the `type` of the dataset provided and the value of the `traj_step` (specified when creating the `Simulation`). If a value for `MD_steps` is not provided, then the minimum number needed will be used automatically.

Additionally, some observables will have an upper limit on the number of MD_steps that can be used in calculating their dependent variable(s). In these cases, the number of `MD_steps` is rounded down to a multiple of this upper limit so that we only run steps that will be useful. For example, if we use 1000 `MD_steps` in calculation, but a value of 2500 is provided, then we will run 2000 steps and use this to calculate the variable twice, without wasting time performing an additional 500 steps.

Finally, start the refinement! `n_steps` has been set to `25` just so you can see what a refinement looks like; it will take many more steps to fully refine a dataset. Bump it up to a higher number when you're ready. Results can also be plotted via the `control.plot_results` method.

In [ ]:
control.refine(n_steps=25)
control.plot_results()